# IT2011 - Artificial Intelligence and Machine Learning
## Progress Review I: Data Preprocessing & Exploratory Data Analysis
### Group ID: `2026-Y2-S1-MET-23`
### Member 4: Abdullah H.F. (IT25102877)
### Assigned Technique: Numerical Cleaning, Outlier Handling & Feature Scaling

---
### 1. Conceptual Framework & Mathematical Background

In natural language sentiment and emotion analysis, non-textual numerical metadata contains crucial predictive signal:
* **User Ratings (`Ratings`):** Numerical evaluation ($1.0 - 10.0$). Positive ratings correlate with `joy` and `optimism`; low ratings correlate with `anger` and `disgust`.
* **Review Length Dynamics (`word_count`, `char_count`):** Reviewers expressing complex negative emotional rants often produce disproportionately large word counts.

#### The Problem of Numerical Outliers
Extreme outliers (e.g., a single review containing 2,500 words vs. the median of 120 words) distort machine learning algorithms:
1. **Distance-based models (SVM, k-NN):** Euclidean distance calculations are dominated by the largest unscaled attribute.
2. **Gradient Descent Optimization (Neural Networks, Logistic Regression):** Unbounded values induce massive gradients, leading to training instability.

#### Outlier Treatment: Tukey's IQR Winsorization (Capping)
Rather than deleting rows—which discards valuable data points and further starves severe minority classes like `surprise` (only 57 examples)—we employ **IQR Capping**:
$$\text{IQR} = Q_3 - Q_1$$
$$\text{Upper Fence} = Q_3 + 1.5 \times \text{IQR}$$
$$\text{Lower Fence} = \max(0, Q_1 - 1.5 \times \text{IQR})$$
Any value $x > \text{Upper Fence}$ is capped at the Upper Fence threshold.

#### Feature Scaling Methodology
* **`RobustScaler`:** Scales features using median and interquartile range:
  $$x_{\text{scaled}} = \frac{x - Q_2}{Q_3 - Q_1}$$
  Because it uses quartiles rather than the sample mean and variance, it is intrinsically immune to remaining extreme values.
* **`MinMaxScaler`:** Bounds user ratings into $[0, 1]$:
  $$x_{\text{norm}} = \frac{x - x_{\min}}{x_{\max} - x_{\min}}$$


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import RobustScaler, MinMaxScaler, StandardScaler

# Set plot aesthetics
sns.set_theme(style="whitegrid", palette="muted")
os.makedirs('../results/eda_visualizations', exist_ok=True)


### 2. Ingesting Dataset & Feature Extraction

In [ ]:
# 1. Load dataset from raw folder
DATA_PATH = '../data/raw/Movies_Reviews_modified_version1.csv'
df = pd.read_csv(DATA_PATH)

# 2. Extract numeric features from text
df['word_count'] = df['Reviews'].astype(str).apply(lambda x: len(x.split()))
df['char_count'] = df['Reviews'].astype(str).apply(len)

print(f"Loaded {df.shape[0]:,} records.")
df[['movie_name', 'Ratings', 'word_count', 'char_count', 'emotion']].head()


### 3. Quantitative Outlier Detection via IQR

In [ ]:
# Display numerical distributions
summary = df[['Ratings', 'word_count', 'char_count']].describe().round(2)
print("Summary Statistics Before Outlier Treatment:")
summary


In [ ]:
# Mathematical Tukey IQR Bounds Calculation
q1 = df['word_count'].quantile(0.25)
q3 = df['word_count'].quantile(0.75)
iqr = q3 - q1
lower_fence = max(0, q1 - 1.5 * iqr)
upper_fence = q3 + 1.5 * iqr

outliers = df[(df['word_count'] < lower_fence) | (df['word_count'] > upper_fence)]
outlier_pct = (len(outliers) / len(df)) * 100

print(f"--- IQR Outlier Report ---")
print(f"Q1 (25th percentile): {q1:.1f} words")
print(f"Q3 (75th percentile): {q3:.1f} words")
print(f"IQR: {iqr:.1f} words")
print(f"Upper Fence: {upper_fence:.1f} words")
print(f"Identified Outliers: {len(outliers):,} ({outlier_pct:.2f}% of dataset)")


### 4. Implementing Winsorization (Capping) & Scaling Pipeline

In [ ]:
# Apply Winsorization: Cap values without deleting samples
df['word_count_capped'] = df['word_count'].clip(lower=lower_fence, upper=upper_fence)

# Apply RobustScaler to capped word counts
robust_scaler = RobustScaler()
df['word_count_robust'] = robust_scaler.fit_transform(df[['word_count_capped']])

# Apply MinMaxScaler to user Ratings to map into [0, 1]
minmax_scaler = MinMaxScaler()
df['rating_normalized'] = minmax_scaler.fit_transform(df[['Ratings']])

print("Sample comparison of transformed features:")
df[['Ratings', 'rating_normalized', 'word_count', 'word_count_capped', 'word_count_robust']].head()


### 5. Individual EDA Visualizations (Viva Presentation)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left Plot: Boxplot showing raw vs. capped word counts
box_data = pd.DataFrame({
    'Raw Word Count': df['word_count'],
    'Capped Word Count (Winsorized)': df['word_count_capped']
})
sns.boxplot(data=box_data, palette=['#e74c3c', '#2ecc71'], ax=axes[0])
axes[0].set_title('Review Word Count: Raw vs. IQR Capped Outliers', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Word Count', fontsize=11)

# Right Plot: User Ratings distribution across emotion categories
sns.violinplot(data=df, x='emotion', y='Ratings', palette='magma', ax=axes[1], inner='quartile')
axes[1].set_title('User Ratings Distribution by Target Emotion', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Emotion Category', fontsize=11)
axes[1].set_ylabel('Numerical Rating (1.0 - 10.0)', fontsize=11)
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
output_fig_path = '../results/eda_visualizations/member4_numerical_outliers_and_ratings.png'
plt.savefig(output_fig_path, dpi=300, bbox_inches='tight')
print(f"Plot saved to: {output_fig_path}")
plt.show()


### 6. Key Findings & Viva Defense Guide (For Abdullah H.F.)

> **Expected Examiner Questions & Answers:**
>
> **Q1: Why did you choose Winsorization (capping) instead of simply dropping outlier rows?**  
> *Answer:* Deleting rows with outlier word counts would discard over 3,000 real user reviews and disproportionately deplete minority classes like `surprise` (which only contains 57 records). Capping retains 100% of data samples while bounding extreme values to prevent gradient instability.
>
> **Q2: Why is `RobustScaler` preferred over `StandardScaler` for review lengths?**  
> *Answer:* `StandardScaler` computes the sample mean $\mu$ and standard deviation $\sigma$, both of which are heavily distorted by extreme values. `RobustScaler` uses the median ($Q_2$) and interquartile range ($Q_3 - Q_1$), ensuring robust scaling unaffected by skewness.
>
> **Q3: What critical pattern does your violin plot reveal?**  
> *Answer:* Numerical ratings have high discriminative power for emotion classification. Positive emotions (`joy`, `optimism`) cluster heavily around ratings 8–10, while negative emotions (`anger`, `disgust`) concentrate at 1–3. This proves that numerical metadata enriches text classifiers.
